# colab_13 — Geneformer CPT eval #3 / detector #2 (catastrophic forgetting)

Did the aggregated CPT run (colab_11 v2) **overwrite the base model's general cell-type knowledge** while adapting to AD glia? This is the other half of the CPT verdict: colab_12 asks whether CPT *helped* the biology (substate, APOE); this notebook asks whether it *hurt* what the model already knew.

**Eval #3 and detector #2 are the same measurement**, read two ways: k-NN cell-type accuracy on a non-AD reference, zero-shot vs post-CPT. The *drop* is scored against eval #3's reporting bands (≤5% acceptable / 5–15% concerning / >15% catastrophic) **and** detector #2's hard gate (>20% drop = overtrain).

**This gates colab_12.** Detector #2 is a go/no-go on the whole run: a substate or APOE "win" in colab_12 that coincides with detector #2 tripping is logged as **overtrain-confounded**, not a win. Section 6 applies that gate to colab_12's recorded verdicts.

**Same two extraction points as colab_12.** The reference is embedded at both `emb_layer=-1` (pipeline readout) and `emb_layer=0` (last layer). Forgetting can hide at one and show at the other — the head-ablation result showed v2's changes concentrate downstream of −1, so reading only −1 could miss damage that L0 reveals.

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment, set run flags

Identical lean Geneformer-only stack as colab_09/11/12 (native numpy-2 base; scGPT not installed). A GPU is required — this notebook embeds the reference four times. `SMOKE` (committed `False`) subsamples the reference *after* its train/test split is drawn, so the whole path can be exercised cheaply; every derived path is `_SMOKE`-suffixed.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .
!pip uninstall -y torchao -q      # unused; hard-raises inside peft dispatch below its floor

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True).stdout.strip()

import torch
assert torch.cuda.is_available(), "no CUDA GPU -- select a GPU runtime before embedding"
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT,
      "| GPU:", torch.cuda.get_device_name(0))

SMOKE   = False
SMOKE_N_PER_TYPE = 60           # cells kept per cell_type when SMOKE
SUFFIX  = "_SMOKE" if SMOKE else ""
SEED    = 0
from datetime import date
TODAY   = date.today().isoformat()
print(f"SMOKE={SMOKE} | output suffix='{SUFFIX}'")

## 2 — Acquire and prepare the non-AD reference

### 2a — Load the reference and validate its schema

The forgetting probe needs an out-of-domain dataset with (i) **raw counts** (Geneformer rank-encodes raw counts) and (ii) trustworthy **cell-type labels** to classify. The reference source is isolated in the config below so it can be swapped without touching anything downstream.

**Default = PBMC** (a labeled, raw-counts peripheral-blood dataset): small, canonical, reproducible, contract-sanctioned. **Known limitation, reported not hidden:** microglia are myeloid, so PBMC's monocyte/DC compartment overlaps the lineage CPT trained on — PBMC can *under-detect* forgetting there; its lymphoid (T/B/NK) types are genuine out-of-domain and carry the probe. **Stricter alternative = Tabula Sapiens** (multi-organ breadth, maximally distant from glia): swap `REF_*` below for a Tabula Sapiens h5ad and the rest of the notebook is unchanged.

`REF_H5AD` is the one value to confirm before running — point it at a real labeled raw-counts h5ad. The asserts below fail loud if raw counts or cell-type labels are missing rather than silently embedding garbage.

In [ ]:
import numpy as np, pandas as pd, anndata as ad, scanpy as sc, scipy.sparse as sp
sc.settings.verbosity = 1

# --- reference config (the one block to edit to swap references) ---
REF_H5AD        = os.path.join(DRIVE_ROOT, "reference", "pbmc_reference.h5ad")  # Kang et al. 2018 (GSE96583)
REF_CELLTYPE_COL = "cell_type"     # obs column holding the cell-type labels to classify
REF_DONOR_COL    = "replicate"     # obs column for donor-disjoint split; set None if absent (Kang 2018: 8 patient donors)
REF_RAW_LAYER    = None            # layer holding raw counts; None => use .X (or .raw if .X is normalized)
REF_NAME         = "pbmc"          # tag used in output filenames / audit
REF_SOURCE_URL   = "https://exampledata.scverse.org/pertpy/kang_2018.h5ad"  # only used to auto-fetch the pbmc default

if not os.path.exists(REF_H5AD) and REF_NAME == "pbmc":
    print("reference not found on Drive -- downloading Kang et al. 2018 (GSE96583, non-AD PBMC reference) ...")
    os.makedirs(os.path.dirname(REF_H5AD), exist_ok=True)
    import urllib.request, shutil
    tmp_path = REF_H5AD + ".part"
    # The host sits behind Cloudflare, which 403s urllib's default `Python-urllib/x.y` User-Agent
    # (verified 2026-07-21: Python-urllib -> 403; browser/curl/requests UAs -> 200). Send a UA
    # explicitly. `urlopen` also replaces `urlretrieve`, which is a deprecated legacy interface.
    req = urllib.request.Request(REF_SOURCE_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(tmp_path, "wb") as fh:
        expected = resp.headers.get("Content-Length")
        shutil.copyfileobj(resp, fh)
    got = os.path.getsize(tmp_path)
    if expected is not None and got != int(expected):
        os.remove(tmp_path)   # never leave a truncated file that a later run would treat as complete
        raise RuntimeError(f"truncated download: got {got} bytes, expected {expected}")
    os.rename(tmp_path, REF_H5AD)   # atomic: a crash mid-download leaves .part, never a truncated "real" file
    print(f"downloaded -> {REF_H5AD} ({got/1e6:.1f} MB)")

assert os.path.exists(REF_H5AD), (
    f"reference not found: {REF_H5AD}. Place a labeled raw-counts non-AD h5ad there "
    "(PBMC default, or a Tabula Sapiens h5ad for the stricter breadth probe).")

ref = sc.read_h5ad(REF_H5AD)
print("reference:", ref.shape)

# resolve raw counts
if REF_RAW_LAYER is not None:
    assert REF_RAW_LAYER in ref.layers, f"layer '{REF_RAW_LAYER}' absent; layers={list(ref.layers)}"
    ref.X = ref.layers[REF_RAW_LAYER].copy()
def _int_frac(X):
    idx = np.random.default_rng(0).choice(X.shape[0], size=min(2000, X.shape[0]), replace=False)
    d = X[idx]; d = d.data if sp.issparse(d) else np.asarray(d).ravel()
    return float(np.mean(np.mod(d, 1) == 0)) if d.size else 0.0
if _int_frac(ref.X) < 0.99 and ref.raw is not None:
    print("`.X` looks normalized -- falling back to `.raw` for raw counts")
    ref = ref.raw.to_adata()
assert _int_frac(ref.X) >= 0.99, "reference `.X` is not raw counts (needed for Geneformer)"

# validate labels
assert REF_CELLTYPE_COL in ref.obs.columns, f"reference lacks cell-type column '{REF_CELLTYPE_COL}'"
ref.obs["cell_type"] = ref.obs[REF_CELLTYPE_COL].astype(str)
assert ref.obs["cell_type"].nunique() >= 5, (
    f"only {ref.obs['cell_type'].nunique()} cell types -- too few for a meaningful forgetting probe")
HAS_DONOR = REF_DONOR_COL is not None and REF_DONOR_COL in ref.obs.columns
ref.obs["donor_id"] = ref.obs[REF_DONOR_COL].astype(str) if HAS_DONOR else "single"
print(f"cell types: {ref.obs['cell_type'].nunique()} | donors: {ref.obs['donor_id'].nunique()} "
      f"(donor-disjoint split: {HAS_DONOR})")
print(ref.obs["cell_type"].value_counts().to_string())

### 2b — Cap per cell type, map genes to the Geneformer vocabulary

The reference is capped per cell type (balanced, up to a ceiling) so no single abundant type dominates the k-NN and so the four embedding passes stay tractable. A stable `ref_index` is assigned as the realignment key. Genes are mapped to Ensembl and vocabulary coverage is reported — unlike colab_12 there is **no APOE hard-fail gate** (that is glia/eval-#2 specific); the forgetting probe only needs broad coverage, so low niche-gene coverage is irrelevant here.

In [ ]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE
import pickle

CAP_PER_TYPE = 3000
rng = np.random.default_rng(SEED)
keep_idx = []
for ct, sub in ref.obs.groupby("cell_type", observed=True):
    pos = np.where(ref.obs["cell_type"].values == ct)[0]
    keep_idx.append(pos if len(pos) <= CAP_PER_TYPE else rng.choice(pos, CAP_PER_TYPE, replace=False))
keep_idx = np.sort(np.concatenate(keep_idx))
ref = ref[keep_idx].copy()
ref.obs["ref_index"] = np.arange(ref.n_obs)
print(f"after per-type cap ({CAP_PER_TYPE}): {ref.n_obs} cells across {ref.obs['cell_type'].nunique()} types")

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f: symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICTIONARY_FILE, "rb") as f: token_dictionary = pickle.load(f)
# reference var_names assumed to be gene SYMBOLS; if they are already Ensembl IDs, set ensembl_id directly.
ref.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in ref.var_names]
in_vocab = ref.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)
frac = float(in_vocab.mean())
print(f"genes in Geneformer vocab: {int(in_vocab.sum())} / {ref.n_vars} ({frac:.1%})")
assert frac >= 0.30, (
    f"only {frac:.1%} of reference genes are tokenizable -- var_names may be Ensembl IDs already "
    "(set ref.var['ensembl_id'] = ref.var_names) or the wrong ID type")

## 3 — Tokenize the reference

### 3a — Tokenize with the same encoding as every other Geneformer notebook

Same V2 / 4096-input encoding and the same `sum_ensembl_ids` RangeIndex monkeypatch colab_11/12 needed. The cell-type, donor, and `ref_index` labels are carried into every tokenized cell so the embeddings realign afterwards.

In [ ]:
from geneformer import TranscriptomeTokenizer
import geneformer.tokenizer as _gftok

ref.obs["n_counts"] = np.asarray(ref.X.sum(axis=1)).ravel()
assert (ref.obs["n_counts"] > 0).all(), "reference cells with zero counts present"

TOK_IN_DIR  = f"/content/gf_ref_token_in{SUFFIX}"
TOK_OUT_DIR = f"/content/gf_ref_token_out{SUFFIX}"
os.makedirs(TOK_IN_DIR, exist_ok=True); os.makedirs(TOK_OUT_DIR, exist_ok=True)
ref.write_h5ad(os.path.join(TOK_IN_DIR, "ref.h5ad"))

CUSTOM_ATTRS = {c: c for c in ["ref_index", "cell_type", "donor_id"]}

_orig_sum = _gftok.sum_ensembl_ids
def _sum_rangeindex_patch(*a, **k):
    r = _orig_sum(*a, **k); r.var.index = pd.RangeIndex(r.n_vars); return r
_gftok.sum_ensembl_ids = _sum_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, f"ref_{REF_NAME}{SUFFIX}", file_format="h5ad")
TOKENIZED_REF = os.path.join(TOK_OUT_DIR, f"ref_{REF_NAME}{SUFFIX}.dataset")
print("tokenized reference ->", TOKENIZED_REF)

## 4 — Embed the reference: zero-shot and CPT, at both extraction points

### 4a — Four embedding passes (base × v2) × (`emb_layer=-1` × `emb_layer=0`)

The frozen base gives the zero-shot embeddings; the base with the v2 LoRA adapter merged in gives the post-CPT embeddings. Each is extracted at both the second-to-last layer (−1, the pipeline readout) and the last layer (0). Four aligned matrices over the same reference cells, each cached to Drive and exists-guarded so a re-run skips the GPU work.

In [ ]:
from peft import PeftModel
from transformers import BertForMaskedLM
from geneformer import EmbExtractor

GF_DIR      = os.path.join(DRIVE_ROOT, "geneformer")
REF_DIR     = os.path.join(DRIVE_ROOT, "reference")
os.makedirs(REF_DIR, exist_ok=True)
MODEL_DIR   = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
ADAPTER_DIR = os.path.join(GF_DIR, "cpt_aggregated_v2_seed0_adapter")   # colab_11 v2 output
assert os.path.exists(MODEL_DIR),   f"base model missing: {MODEL_DIR}"
assert os.path.exists(ADAPTER_DIR), f"v2 LoRA adapter missing on Drive: {ADAPTER_DIR}"

EMB_LABELS = ["ref_index", "cell_type", "donor_id"]
LAYER_TAG  = {-1: "L-1", 0: "L0"}

def _ref_emb_path(model_tag, layer):
    # cap/seed baked into the filename so a re-run with changed params can never silently
    # load a stale cached embedding keyed only on model_tag/layer
    return os.path.join(REF_DIR,
        f"ref_{REF_NAME}_{model_tag}_{LAYER_TAG[layer].replace('-','m')}_cap{CAP_PER_TYPE}_seed{SEED}{SUFFIX}.h5ad")

def _extract(model_dir, model_tag, layer):
    out = _ref_emb_path(model_tag, layer)
    if os.path.exists(out):
        print(f"{model_tag} {LAYER_TAG[layer]} exists, loading:", out)
        a = sc.read_h5ad(out)
    else:
        ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                          max_ncells=None, emb_layer=layer, emb_label=EMB_LABELS,
                          forward_batch_size=64, nproc=4)
        work = f"/content/gf_ref_emb_{model_tag}_{LAYER_TAG[layer]}{SUFFIX}"; os.makedirs(work, exist_ok=True)
        df = ee.extract_embs(model_dir, TOKENIZED_REF, work, f"ref_{model_tag}_{LAYER_TAG[layer]}")
        emb_cols = [c for c in df.columns if c not in EMB_LABELS]
        df = df.set_index("ref_index").reindex(ref.obs["ref_index"].values)
        assert df[emb_cols].notna().all().all(), f"{model_tag} {LAYER_TAG[layer]}: rows missing after realignment"
        a = ad.AnnData(X=df[emb_cols].to_numpy(dtype=np.float32),
                       obs=ref.obs[EMB_LABELS].reset_index(drop=True))
        a.write_h5ad(out)
        print(f"{model_tag} {LAYER_TAG[layer]} -> {out}  {a.shape}")
    return a.X.astype(np.float32)

# zero-shot: frozen base at both layers
EMB = {}
for layer in (-1, 0):
    EMB[("zeroshot", LAYER_TAG[layer])] = _extract(MODEL_DIR, "zeroshot", layer)

# CPT: merge v2 adapter once, then both layers
MERGED_DIR = f"/content/gf_v2_merged{SUFFIX}"
if not os.path.exists(os.path.join(MERGED_DIR, "config.json")):
    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    os.makedirs(MERGED_DIR, exist_ok=True)
    merged.save_pretrained(MERGED_DIR)
    base.config.to_json_file(os.path.join(MERGED_DIR, "config.json"))
for layer in (-1, 0):
    EMB[("cpt", LAYER_TAG[layer])] = _extract(MERGED_DIR, "cpt", layer)

for k, v in EMB.items(): print(k, v.shape)
assert len({v.shape for v in EMB.values()}) == 1, "reference embedding matrices differ in shape"

## 5 — Eval #3 / Detector #2: k-NN cell-type forgetting

### 5a — Reference split and composition (donor-disjoint where possible; rare types excluded)

A held-out split of the reference: donor-disjoint if the reference carries donors, else stratified by cell type. Because zero-shot and CPT are scored on the *same* split, any residual leakage affects both equally and cancels in the drop. Cell types too rare in aggregate are dropped before the split. Separately, cell types that end up too thin specifically *in the held-out test split* — a real risk with a handful of donors and uneven per-donor composition, since a donor-disjoint split can't stratify by cell type — are excluded from k-NN scoring after the split, so an unreliable per-class accuracy estimate can never inject noise into the drop that feeds detector #2's gate. A type the probe cannot reliably see is not evidence of forgetting either way.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit

ct = ref.obs["cell_type"].to_numpy()
dn = ref.obs["donor_id"].to_numpy()

# --- drop rare cell types before splitting ---
MIN_CELLS_PER_TYPE = 50
vc = pd.Series(ct).value_counts()
keep_types = set(vc[vc >= MIN_CELLS_PER_TYPE].index)
dropped = sorted(set(vc.index) - keep_types)
keep = np.isin(ct, list(keep_types))
print(f"cell types kept: {len(keep_types)} | dropped (<{MIN_CELLS_PER_TYPE} cells): {dropped}")

idx = np.where(keep)[0]
if ref.obs["donor_id"].nunique() > 3:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_rel, te_rel = next(gss.split(idx, ct[idx], groups=dn[idx]))
    split_kind = "donor-disjoint"
else:
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=SEED)
    tr_rel, te_rel = next(sss.split(idx, ct[idx]))
    split_kind = "stratified-by-celltype"
train_idx, test_idx = idx[tr_rel], idx[te_rel]

# SMOKE: thin each cell type AFTER the split is drawn
if SMOKE:
    def _thin(ix):
        out = []
        for c in keep_types:
            p = ix[ct[ix] == c]
            out.append(p if len(p) <= SMOKE_N_PER_TYPE else rng.choice(p, SMOKE_N_PER_TYPE, replace=False))
        return np.sort(np.concatenate(out))
    train_idx, test_idx = _thin(train_idx), _thin(test_idx)

print(f"split: {split_kind} | train {len(train_idx)} / test {len(test_idx)} cells")
print("test cell-type counts:")
print(pd.Series(ct[test_idx]).value_counts().to_string())

# --- exclude cell types too thin in the TEST split to score reliably ---
# The MIN_CELLS_PER_TYPE floor above only guards aggregate counts, drawn before the split.
# A donor-disjoint split over this few donors (and per-donor composition this uneven) can
# still leave a type with a handful of test cells even though it cleared that floor.
# balanced_accuracy_score weights every class equally regardless of size, so a thin class
# injects noise into the zero-shot-vs-CPT drop that does NOT cancel between the two --
# exactly the kind of instability a hard pass/fail gate (detector #2) shouldn't be exposed to.
MIN_TEST_PER_TYPE = 20
test_vc = pd.Series(ct[test_idx]).value_counts()
underpowered_types = sorted(test_vc[test_vc < MIN_TEST_PER_TYPE].index)
if underpowered_types:
    print(f"\nWARNING: {len(underpowered_types)} type(s) have <{MIN_TEST_PER_TYPE} test cells "
          f"after the split -- excluding from k-NN scoring: {underpowered_types}")
    train_idx = train_idx[~np.isin(ct[train_idx], underpowered_types)]
    test_idx  = test_idx[~np.isin(ct[test_idx],  underpowered_types)]
scored_types = sorted(keep_types - set(underpowered_types))
print(f"\nfinal: train {len(train_idx)} / test {len(test_idx)} cells, {len(scored_types)} types scored: {scored_types}")

### 5b — k-NN cell-type accuracy, zero-shot vs CPT, at both layers → forgetting

For each extraction point, a k-NN is fit on the train cells' embedding to predict cell type and scored (balanced accuracy across types) on the held-out cells — once on the zero-shot embedding, once on the CPT embedding. The **drop** = zero-shot − CPT (positive = forgetting) is scored against eval #3's bands and detector #2's gate. Reading both layers shows whether any forgetting sits at the pipeline readout (−1) or concentrates in the last layer (0) where v2's changes are known to live.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

def band_forget(drop_pp):        # eval #3 reporting bands (drop = zs - cpt, positive = worse)
    if drop_pp > 15: return "catastrophic"
    if drop_pp > 5:  return "concerning"
    return "acceptable"
def gate_detector2(drop_pp):     # detector #2 hard gate
    return "OVERTRAIN" if drop_pp > 20 else "pass"

def knn_bacc(X):
    sc_ = StandardScaler().fit(X[train_idx])
    knn = KNeighborsClassifier(n_neighbors=15).fit(sc_.transform(X[train_idx]), ct[train_idx])
    return balanced_accuracy_score(ct[test_idx], knn.predict(sc_.transform(X[test_idx])))

if underpowered_types:
    print(f"NOTE: {len(underpowered_types)} type(s) excluded from every layer's scoring for thin "
          f"test representation (<{MIN_TEST_PER_TYPE} cells): {underpowered_types}\n")

EVAL3 = {}
for layer in ("L-1", "L0"):
    acc_zs  = knn_bacc(EMB[("zeroshot", layer)])
    acc_cpt = knn_bacc(EMB[("cpt", layer)])
    drop = (acc_zs - acc_cpt) * 100
    EVAL3[layer] = {"acc_zeroshot": round(float(acc_zs), 4), "acc_cpt": round(float(acc_cpt), 4),
                    "drop_pp": round(float(drop), 2), "eval3_verdict": band_forget(drop),
                    "detector2_gate": gate_detector2(drop),
                    "n_types_scored": len(scored_types), "excluded_types": underpowered_types}
    print(f"{layer}: zero-shot {acc_zs:.3f} -> CPT {acc_cpt:.3f} | drop {drop:+.2f} pp | "
          f"eval#3 {band_forget(drop)} | detector#2 {gate_detector2(drop)}")

# primary gate = pipeline readout (-1); flag if L0 is materially worse than -1
GATE_PRIMARY = EVAL3["L-1"]["detector2_gate"]
L0_WORSE = EVAL3["L0"]["drop_pp"] - EVAL3["L-1"]["drop_pp"] > 5
print(f"\ndetector #2 (primary, L-1): {GATE_PRIMARY}"
      + ("  | NOTE: L0 forgetting exceeds L-1 by >5pp -- deeper-layer damage the readout hides" if L0_WORSE else ""))

## 6 — Summary, gate on colab_12, and handoff

### 6a — Apply detector #2 to colab_12's wins, append the audit trace, print commit commands

Detector #2's verdict is applied back to colab_12: if it tripped, any eval #1/#2 win recorded under `geneformer_cpt_evals` is re-labelled **overtrain-confounded**. If colab_12 has not been run yet, that is noted and the gate is deferred. The forgetting results are written under `geneformer_cpt_forgetting`. A `SMOKE` run writes nothing to the audit trace.

In [ ]:
import json, shlex

print("=== EVAL #3 / DETECTOR #2 (forgetting on non-AD reference: %s) ===" % REF_NAME)
for layer, r in EVAL3.items():
    print(f"  {layer:4s}: {r['acc_zeroshot']:.3f} -> {r['acc_cpt']:.3f} | drop {r['drop_pp']:+.2f}pp "
          f"| eval#3 {r['eval3_verdict']} | detector#2 {r['detector2_gate']}")

AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
with open(AUDIT_REPORT_PATH) as f:
    report = json.load(f)

# gate colab_12's recorded wins
overtrain = GATE_PRIMARY == "OVERTRAIN"
gate_note = []
if "geneformer_cpt_evals" in report:
    e = report["geneformer_cpt_evals"]
    wins = []
    for blk in ("eval1_substate_probe", "eval2_apoe_recovery"):
        for key, r in e.get(blk, {}).items():
            for vk in ("verdict", "knn_verdict"):
                if r.get(vk) in ("meaningful", "decisive"):
                    wins.append(f"{blk}:{key}:{vk}={r[vk]}")
    if overtrain and wins:
        gate_note = [f"OVERTRAIN-CONFOUNDED (detector#2 tripped): {w}" for w in wins]
        print("\n[GATE] detector #2 tripped -- colab_12 wins re-labelled overtrain-confounded:")
        for w in wins: print("   -", w)
    elif wins:
        gate_note = [f"win stands (detector#2 pass): {w}" for w in wins]
        print(f"\n[GATE] detector #2 pass -- {len(wins)} colab_12 win(s) stand.")
    else:
        print("\n[GATE] colab_12 recorded no eval wins to gate.")
else:
    gate_note = ["colab_12 (geneformer_cpt_evals) not yet in audit -- apply gate once it runs"]
    print("\n[GATE] colab_12 evals not yet recorded -- gate deferred until it runs.")

if SMOKE:
    print("\n[SMOKE] audit trace NOT written (plumbing run).")
else:
    report["geneformer_cpt_forgetting"] = {
        "status": "computed", "date": TODAY, "reads_run": "geneformer_cpt_aggregated_v2",
        "reference": REF_NAME, "reference_file": os.path.relpath(REF_H5AD, DRIVE_ROOT),
        "n_ref_cells": int(ref.n_obs), "n_cell_types": int(ref.obs["cell_type"].nunique()),
        "geneformer_commit": GENEFORMER_COMMIT,
        "extraction_points": {"L-1": "second-to-last", "L0": "last layer"},
        "eval3_detector2": EVAL3, "detector2_primary_gate": GATE_PRIMARY,
        "colab_12_gate": gate_note,
        "note": ("eval#3 and detector#2 are one k-NN read two ways. PBMC's myeloid overlap with microglia "
                 "may under-detect forgetting; Tabula Sapiens is the stricter breadth swap."),
    }
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("\naudit trace appended ->", AUDIT_REPORT_PATH)
    rel = os.path.relpath(AUDIT_REPORT_PATH, REPO_PATH)
    print("\n=== Commit + push (from WSL) ===")
    print(f"  git add {shlex.quote(rel)}")
    print("  git commit -m 'colab_13: Geneformer CPT eval #3 / detector #2 (forgetting) at L-1 and L0'")
    print("  git push")